<a href="https://colab.research.google.com/github/dantruongnv/AI-Email-Spam-Detector/blob/main/AI_Spam_Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 29.5 MB/s eta 0:00:00
  Attempting uninstall: google-api-python-client
    Found existing installation: google-api-python-client 2.198.0
    Uninstalling google-api-python-client-2.198.0:
      Successfully uninstalled google-api-python-client-2.198.0


In [ ]:
from google.colab import files

print("Hãy chọn file credentials.json từ máy tính của bạn:")
uploaded = files.upload()

Hãy chọn file credentials.json từ máy tính của bạn:


Saving credentials.json to credentials.json


In [ ]:
import os.path
from googleapiclient.discovery import build
from oauth2client.client import GoogleCredentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials

SCOPES = ['https://www.googleapis.com/auth/gmail.modify']

def get_unread_emails():
    creds = None
    # 1. Kiểm tra nếu đã có token đăng nhập cũ
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    # 2. Nếu chưa có hoặc token hết hạn thì tiến hành đăng nhập
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            # Chạy flow đăng nhập tương thích Colab
            flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
            flow.redirect_uri = 'https://localhost'

            auth_url, _ = flow.authorization_url(prompt='consent')

            print("1. Click vào liên kết sau để cấp quyền Google:")
            print(auth_url)
            print("\n2. Sau khi bấm 'Cho phép', trình duyệt sẽ chuyển sang trang 'localhost' báo lỗi không tìm thấy trang.")
            print("   -> BẠN CHỈ CẦN COPY TOÀN BỘ ĐƯỜNG DẪN URL TRÊN THANH ĐỊA CHỈ TRÌNH DUYỆT ĐÓ DÁN VÀO BÊN DƯỚI.")

            redirect_response = input("\n3. Dán toàn bộ đường dẫn URL trang lỗi vào đây: ")

            flow.fetch_token(authorization_response=redirect_response)
            creds = flow.credentials

        with open('token.json', 'w') as token:
            token.write(creds.to_json())

    # 3. Kết nối Gmail API
    service = build('gmail', 'v1', credentials=creds)

    # Lấy 5 email CHƯA ĐỌC trong Inbox
    results = service.users().messages().list(userId='me', q='is:unread', maxResults=5).execute()
    messages = results.get('messages', [])

    emails_data = []
    if not messages:
        print("\n[THÔNG BÁO]: Không có email nào chưa đọc trong Inbox.")
    else:
        print(f"\n[THÀNH CÔNG]: Đã kết nối Gmail và lấy {len(messages)} email chưa đọc!")
        for message in messages:
            msg = service.users().messages().get(userId='me', id=message['id']).execute()
            snippet = msg.get('snippet', '')
            emails_data.append({'id': message['id'], 'snippet': snippet})

    return service, emails_data

# Chạy đăng nhập
service, unread_emails = get_unread_emails()

1. Click vào liên kết sau để cấp quyền Google:
https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=124837651247-b949figr5n1v4bkksao9ddsj98tkmjoi.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Flocalhost&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.modify&state=61zELBXWyNLfKGIFNAwS2v1bHu3B7i&code_challenge=XdW2NnNDJICgyeP2Sv-LZ4QGFKy_3ugSACBDI1m9rsM&code_challenge_method=S256&prompt=consent&access_type=offline

2. Sau khi bấm 'Cho phép', trình duyệt sẽ chuyển sang trang 'localhost' báo lỗi không tìm thấy trang.
   -> BẠN CHỈ CẦN COPY TOÀN BỘ ĐƯỜNG DẪN URL TRÊN THANH ĐỊA CHỈ TRÌNH DUYỆT ĐÓ DÁN VÀO BÊN DƯỚI.

3. Dán toàn bộ đường dẫn URL trang lỗi vào đây: https://localhost/?state=61zELBXWyNLfKGIFNAwS2v1bHu3B7i&iss=https://accounts.google.com&code=4/0ATsMZqDxE8Alshz9sAm-1lWm2jdn057HkVYPLGl6objrRwJAv_vdZG9PKw_YLZ3g0BSa6g&scope=https://www.googleapis.com/auth/gmail.modify

[THÀNH CÔNG]: Đã kết nối Gmail và lấy 5 email chưa đọc!


In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. TẢI VÀ GIẢI NÉN DATASET CHUẨN (SMS Spam Collection Dataset - 5,572 emails/messages)
print("1. Đang tải bộ dữ liệu Spam Dataset...")
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "smsspamcollection.zip"

if not os.path.exists("SMSSpamCollection"):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("   -> Tải và giải nén Dataset thành công!")

# 2. ĐỌC VÀ TIỀN XỬ LÝ DỮ LIỆU
df = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'text'])
print(f"   -> Tổng số lượng mẫu dữ liệu huấn luyện: {len(df)} dòng")

X = df['text']
y = df['label']

# SỬA LỖI TẠI ĐÂY: Dùng test_size thay vì test_split_size
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. TRÍCH XUẤT ĐẶC TRƯNG VỚI TF-IDF VÀ HUẤN LUYỆN MÔ HÌNH NAIVE BAYES
print("\n2. Đang huấn luyện mô hình AI (Naive Bayes + TF-IDF)...")
vectorizer = TfidfVectorizer(stop_words='english', lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Đánh giá nhanh độ chính xác
X_test_vec = vectorizer.transform(X_test)
acc = accuracy_score(y_test, model.predict(X_test_vec))
print(f"   -> Huấn luyện xong! Độ chính xác của mô hình trên tập test: {acc * 100:.2f}%\n")

# 4. TIẾN HÀNH PHÂN LOẠI EMAIL THỰC TẾ TỪ GMAIL API (BƯỚC 3)
print("=" * 70)
print("=== BẮT ĐẦU PHÂN LOẠI HÒM THƯ GMAIL CỦA BẠN BẰNG AI ===")
print("=" * 70)

if 'unread_emails' not in locals() or not unread_emails:
    print("[THÔNG BÁO]: Không tìm thấy danh sách 'unread_emails' từ Bước 3 hoặc hòm thư không có thư mới chưa đọc.")
else:
    print(f"Tìm thấy {len(unread_emails)} email chưa đọc từ Hộp thư đến (Inbox).\n")

    for index, item in enumerate(unread_emails, 1):
        email_text = item['snippet']
        email_id = item['id']

        # Biến đổi văn bản email bằng Vectorizer đã học từ Dataset 5,500 mẫu
        text_vector = vectorizer.transform([email_text])

        # Dự đoán
        prediction = model.predict(text_vector)[0]
        confidence = model.predict_proba(text_vector).max() * 100

        print(f"[{index}] Nội dung email: '{email_text[:80]}...'")

        if prediction == 'spam':
            print(f"   -> KẾT QUẢ AI: [ SPAM / THƯ RÁC ] (Độ tin cậy: {confidence:.1f}%)")

            # Thực thi chuyển thư rác vào SPAM trên Gmail
            service.users().messages().modify(
                userId='me',
                id=email_id,
                body={'addLabelIds': ['SPAM'], 'removeLabelIds': ['INBOX']}
            ).execute()
            print("   -> HÀNH ĐỘNG: Đã tự động chuyển thư này vào thư mục SPAM!")
        else:
            print(f"   -> KẾT QUẢ AI: [ HAM / THƯ THƯỜNG ] (Độ tin cậy: {confidence:.1f}%)")
            print("   -> HÀNH ĐỘNG: Giữ nguyên thư trong Hộp thư đến (Inbox).")

        print("-" * 70)

1. Đang tải bộ dữ liệu Spam Dataset...
   -> Tổng số lượng mẫu dữ liệu huấn luyện: 5572 dòng

2. Đang huấn luyện mô hình AI (Naive Bayes + TF-IDF)...
   -> Huấn luyện xong! Độ chính xác của mô hình trên tập test: 97.85%

=== BẮT ĐẦU PHÂN LOẠI HÒM THƯ GMAIL CỦA BẠN BẰNG AI ===
Tìm thấy 5 email chưa đọc từ Hộp thư đến (Inbox).

[1] Nội dung email: 'Ai đó mới đăng nhập tài khoản dantruongnv06@gmail.com Chúng tôi phát hiện thấy m...'
   -> KẾT QUẢ AI: [ HAM / THƯ THƯỜNG ] (Độ tin cậy: 82.8%)
   -> HÀNH ĐỘNG: Giữ nguyên thư trong Hộp thư đến (Inbox).
----------------------------------------------------------------------
[2] Nội dung email: 'Chúng tôi đã khôi phục thành công tài khoản của bạn dantruongnv06@gmail.com Chào...'
   -> KẾT QUẢ AI: [ HAM / THƯ THƯỜNG ] (Độ tin cậy: 62.4%)
   -> HÀNH ĐỘNG: Giữ nguyên thư trong Hộp thư đến (Inbox).
----------------------------------------------------------------------
[3] Nội dung email: 'Generate, refine, and export in minutes Just tap a chat to st